# Minimal example of Automation

Each node generates a random integer between 1 and 10. A layer passes if the sum of the results is greater than 20.

In [ ]:
import random

import attrs

from laboneq._automation import Automation, AutomationLayer
from laboneq._automation.logic import FixedParameterUpdate
from laboneq._automation.status import AutomationStatus as Status
from laboneq.core.utilities.dsl_dataclass_decorator import classformatter

In [ ]:
@classformatter
@attrs.define
class ExampleLayer(AutomationLayer):
    def run_executable(self, automation: "Automation") -> dict[str, int]:
        # Set node statuses (pre run)
        for node in self.nodes.values():
            node.status = Status.RUNNING

        # Run function
        parameters = automation.automation_parameters[self.key]
        results = self.function(self.node_keys, parameters)
        self.results = results

        # Set node statuses (post run)
        for node in self.nodes.values():
            node.status = Status.PASSED if sum(results.values()) > 20 else Status.FAILED

        return results

In [ ]:
auto_params = {
    "l1": {"n1": {"seed": 1}, "n2": {"seed": 2}, "n3": {"seed": 3}, "n4": {"seed": 4}},
    "l2": {"n1": {"seed": 1}, "n2": {"seed": 2}, "n3": {"seed": 3}, "n4": {"seed": 4}},
    "l3": {"n1": {"seed": 1}, "n2": {"seed": 2}, "n3": {"seed": 3}, "n4": {"seed": 4}},
}
auto = Automation(automation_parameters=auto_params, name="example")
auto.plot();

In [ ]:
def generate_random_ints(node_keys: list[str], parameters: dict) -> list[int]:
    results = {}
    for node_key in node_keys:
        seed = parameters[node_key]["seed"]
        random.seed(seed)
        results[node_key] = random.randint(1, 10)
    return results


l1 = ExampleLayer(
    generate_random_ints, ["n1", "n2", "n3", "n4"], key="l1", depends_on={"root"}
)
auto.add_layer(l1)
l2 = ExampleLayer(
    generate_random_ints, ["n1", "n2", "n3", "n4"], key="l2", depends_on={"l1"}
)
auto.add_layer(l2)
l3 = ExampleLayer(
    generate_random_ints, ["n1", "n2", "n3", "n4"], key="l3", depends_on={"l2"}
)
auto.add_layer(l3)
auto.plot();

In [ ]:
auto.run()
auto.plot();

In [ ]:
auto["l1"].results

In [ ]:
auto.reset()
auto.plot();

In [ ]:
auto.get_layer("l1").parameters

In [ ]:
for layer in auto.layers(include_root=False):
    layer.logic = FixedParameterUpdate(
        new_layer_key=layer.key,
        parameter_changes={
            "n1": {"seed": 1},
            "n2": {"seed": 1},
            "n3": {"seed": 1},
            "n4": {"seed": 1},
        },
    )

In [ ]:
auto.run()
auto.plot();

In [ ]:
auto.get_layer("l1").fail_count

In [ ]:
auto["l1"].results

In [ ]:
auto.reset()
auto.plot();

In [ ]:
auto["l1"]

In [ ]:
auto.run_from_node("l1_n1")
auto.plot();

In [ ]:
auto.get_layer("l1").results

In [ ]:
auto["l1"]["n1"]